In [1]:
import scanpy as sc
import anndata as ad
import numpy as np
from sklearn.utils import shuffle
from xgboost import XGBClassifier
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, confusion_matrix, precision_score, recall_score
import shap
import pandas as pd
import joblib

/home/michal_sieczczynski/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
adata_train_healthy = sc.read_h5ad("./ready_data/train_healthy_fixed.h5ad")
adata_train_tumor = sc.read_h5ad("./ready_data/train_tumor_fixed.h5ad")

adata_train_healthy.obs['label'] = 0
adata_train_tumor.obs['label'] = 1

adata_train = ad.concat([adata_train_healthy, adata_train_tumor], join='inner')

X_train = adata_train.X
y_train = adata_train.obs['label'].values

X_train, y_train = shuffle(X_train, y_train, random_state=42)

In [3]:
adata_test_healthy = sc.read_h5ad("./ready_data/test_healthy_fixed.h5ad")
adata_test_tumor = sc.read_h5ad("./ready_data/test_tumor_fixed.h5ad")

adata_test_healthy.obs['label'] = 0
adata_test_tumor.obs['label'] = 1

adata_test = ad.concat([adata_test_healthy, adata_test_tumor], join='inner')

X_test = adata_test.X
y_test = adata_test.obs['label'].values

In [4]:
adata_val_healthy = sc.read_h5ad("./ready_data/val_healthy_fixed.h5ad")
adata_val_tumor = sc.read_h5ad("./ready_data/val_tumor_fixed.h5ad")

adata_val_healthy.obs['label'] = 0
adata_val_tumor.obs['label'] = 1

adata_val = ad.concat([adata_val_healthy, adata_val_tumor], join='inner')

X_val = adata_val.X
y_val = adata_val.obs['label'].values

In [5]:
adata_test_ctc = sc.read_h5ad("./ready_data/ctc_normalized_data_fixed.h5ad")
adata_test_ctc.obs['label'] = 1

X_test_ctc = adata_test_ctc.X
y_test_ctc = adata_test_ctc.obs['label'].values

In [6]:
xgboost_object = XGBClassifier(max_depth=29, learning_rate=0.164985, n_estimators=95, 
                            reg_alpha=8.553876, reg_lambda=0.17816, min_child_weight=0.001009,
                            seed=42)
# params_learning_rate                                       0.164985
# params_max_depth                                                 29
# params_min_child_weight                                    0.001009
# params_n_estimators                                              95
# params_reg_alpha                                           8.553876
# params_reg_lambda                                           0.17816


xgboost_object.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.164985,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=29, max_leaves=None,
              min_child_weight=0.001009, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=95, n_jobs=None,
              num_parallel_tree=None, random_state=None, ...)

In [8]:
gene_names = adata_test_ctc.var_names.to_numpy()

importance_df = pd.DataFrame({
    "gene": gene_names,
    "importance": xgboost_object.feature_importances_
})

importance_df = importance_df[
    importance_df["importance"] > 0
].sort_values(
    "importance",
    ascending=False
)
print(importance_df)

          gene    importance
8995     RPL13  6.045377e-01
3130    EEF1A1  3.041942e-01
9031     RPL41  6.459764e-02
8991     RPL10  2.648918e-02
9039     RPLP0  1.235656e-04
9002     RPL19  1.704614e-05
2          A2M  1.064710e-05
229    ADIPOR1  1.000072e-05
10947  TMEM236  5.973743e-06
22       ABCA4  5.519796e-06
145       ACTB  5.021319e-06
126       ACP5  1.350961e-06
940       AZU1  1.350961e-06
3924      FTH1  2.936520e-07
9040     RPLP1  2.915110e-07
9831    SLC4A1  1.755812e-07
3189      EIF1  7.641771e-08
941        B2M  3.543452e-08
5391      KLF1  3.085297e-08
457     ANKLE1  1.159891e-08
4554     HEMGN  2.187787e-09
130     ACSBG1  9.847640e-10


In [7]:
joblib.dump(xgboost_object, "xgboost_final_model_fixed.pkl")

['xgboost_final_model_fixed.pkl']

In [8]:
y_pred = xgboost_object.predict(X_val)
print(f"Balanced accuracy: {balanced_accuracy_score(y_val, y_pred)}")
print(f"Precision for Tumor: {precision_score(y_val, y_pred, pos_label=1)}")
print(f"Precision for PBMC: {precision_score(y_val, y_pred, pos_label=0)}")
print(f"ROC AUC: {roc_auc_score(y_val, xgboost_object.predict_proba(X_val)[:,1])}")

cm = confusion_matrix(y_val, y_pred)
print(cm)

Balanced accuracy: 1.0
Precision for Tumor: 1.0
Precision for PBMC: 1.0
ROC AUC: 1.0
[[60488     0]
 [    0 24773]]


In [9]:
y_pred = xgboost_object.predict(X_test)
print(f"Balanced accuracy: {balanced_accuracy_score(y_test, y_pred)}")
print(f"Precision for Tumor: {precision_score(y_test, y_pred, pos_label=1)}")
print(f"Precision for PBMC: {precision_score(y_test, y_pred, pos_label=0)}")
print(f"ROC AUC: {roc_auc_score(y_test, xgboost_object.predict_proba(X_test)[:,1])}")

cm = confusion_matrix(y_test, y_pred)
print(cm)

Balanced accuracy: 1.0
Precision for Tumor: 1.0
Precision for PBMC: 1.0
ROC AUC: 1.0
[[33783     0]
 [    0 24160]]


In [10]:
y_pred = xgboost_object.predict(X_test_ctc)
print(f"Balanced accuracy: {balanced_accuracy_score(y_test_ctc, y_pred)}")
print(f"Recall: {recall_score(y_test_ctc, y_pred, pos_label=1)}")

cm = confusion_matrix(y_test_ctc, y_pred)
print(cm)

Balanced accuracy: 0.9855491329479769
Recall: 0.9855491329479769
[[   0    0]
 [  15 1023]]


/home/michal_sieczczynski/miniconda3/envs/sequencing/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2480: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [11]:
gene_names = adata_test_ctc.var_names.tolist()
explainer = shap.TreeExplainer(xgboost_object)
shap_values = explainer(X_test).values

global_shap = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({
    "gene": gene_names,
    "SHAP": global_shap
})

shap_df = shap_df[shap_df["SHAP"] != 0]
shap_df = shap_df.sort_values("SHAP", ascending=False)

shap_df.to_csv("feature_importance/xgboost_shap_values_tumor_test_fixed.csv", index=False)

In [12]:
shap_values = explainer(X_test_ctc).values

global_shap = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({
    "gene": gene_names,
    "SHAP": global_shap
})

shap_df = shap_df[shap_df["SHAP"] != 0]
shap_df = shap_df.sort_values("SHAP", ascending=False)

shap_df.to_csv("feature_importance/xgboost_shap_values_ctc_test_fixed.csv", index=False)